# 🚀 Water Meter AI - Local Training

**Author:** Arsenius Purbandono  
**Objective:** Train YOLOv8-OBB model on local machine

---

## 1. Setup Environment

In [ ]:
# Check GPU availability
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

In [ ]:
# Import required libraries
import sys
from pathlib import Path

# Add project root to path
project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.append(str(project_root))

from ultralytics import YOLO
from src.utils.config import Config
from src.utils.logger import setup_logger, get_logger

# Setup logging
setup_logger(level='INFO')
logger = get_logger(__name__)

print("✅ Environment setup complete!")

## 2. Load Configuration

In [ ]:
# Load training config
config_path = project_root / 'configs' / 'train_config.yaml'
config = Config(str(config_path))

print("Configuration loaded:")
print(f"  Model: {config.get('model.type')}")
print(f"  Epochs: {config.get('training.epochs')}")
print(f"  Batch Size: {config.get('training.batch_size')}")
print(f"  Image Size: {config.get('data.img_size')}")

## 3. Dataset Verification

In [ ]:
# Check dataset structure
data_yaml = project_root / config.get('data.yaml_path')
print(f"Dataset YAML: {data_yaml}")
print(f"Exists: {data_yaml.exists()}")

if data_yaml.exists():
    import yaml
    with open(data_yaml, 'r') as f:
        data_config = yaml.safe_load(f)
    print("\nDataset configuration:")
    print(f"  Train images: {data_config.get('train', 'N/A')}")
    print(f"  Val images: {data_config.get('val', 'N/A')}")
    print(f"  Test images: {data_config.get('test', 'N/A')}")
    print(f"  Classes: {data_config.get('nc', 'N/A')}")
    print(f"  Names: {data_config.get('names', 'N/A')}")

## 4. Initialize Model

In [ ]:
# Initialize YOLOv8-OBB model
model_type = config.get('model.type', 'yolov8n-obb')
print(f"Initializing {model_type}...")

model = YOLO(f"{model_type}.pt")
print("✅ Model initialized!")
print(f"Model summary: {model.model}")

## 5. Training

In [ ]:
# Start training
results = model.train(
    data=str(data_yaml),
    epochs=config.get('training.epochs', 100),
    batch=config.get('training.batch_size', 16),
    imgsz=config.get('data.img_size', 512),
    patience=config.get('training.patience', 20),
    
    # Optimizer
    optimizer=config.get('training.optimizer', 'AdamW'),
    lr0=config.get('training.lr0', 0.01),
    lrf=config.get('training.lrf', 0.01),
    
    # Augmentation
    degrees=config.get('augmentation.degrees', 15.0),
    
    # Hardware
    device='',  # Auto-detect
    workers=config.get('data.workers', 8),
    amp=config.get('hardware.amp', True),
    
    # Logging
    project=config.get('logging.project', 'water-meter-ai'),
    name=config.get('logging.name', 'exp'),
    plots=True,
    verbose=True,
)

## 6. Validation

In [ ]:
# Validate the trained model
metrics = model.val()

print("\n📊 Validation Metrics:")
print(f"  mAP50: {metrics.box.map50:.4f}")
print(f"  mAP50-95: {metrics.box.map:.4f}")
print(f"  Precision: {metrics.box.mp:.4f}")
print(f"  Recall: {metrics.box.mr:.4f}")

## 7. Export to TFLite

In [ ]:
# Export to TFLite format
print("Exporting to TFLite (INT8 Quantized)...")

tflite_path = model.export(
    format='tflite',
    imgsz=512,
    int8=True,
    optimize=True,
)

print(f"✅ TFLite model saved: {tflite_path}")

# Check model size
tflite_file = Path(tflite_path)
if tflite_file.exists():
    size_mb = tflite_file.stat().st_size / (1024 * 1024)
    print(f"📦 Model size: {size_mb:.2f} MB")

## 8. Test Inference

In [ ]:
# Test inference on sample image
test_image = project_root / 'test' / 'images'
test_images = list(test_image.glob('*.jpg'))[:5]  # Take 5 samples

if test_images:
    results = model(test_images[0])
    results[0].show()  # Display results
else:
    print("No test images found!")

## ✅ Training Complete!

**Next Steps:**
1. Check training results in `runs/train/exp/`
2. Analyze confusion matrix and metrics
3. Test exported TFLite model on mobile device
4. Integrate into Flutter app